# Dependencies & Libraries

In [23]:
from google.cloud import bigquery
from google.colab import auth
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, accuracy_score)
from sklearn.impute import SimpleImputer

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
!pip install keras-tuner
import keras_tuner as kt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.0 MB/s eta 0:00:00


# Load Data from BigQuery

In [7]:
auth.authenticate_user()
PROJECT_ID = 'crediloan'
DATASET_ID = 'Bank_Data'
TBL_APPLICATION = 'tbl_applications'
TBL_BORROWS = 'tbl_borrowes'
TBL_EMPLOYMENT = 'tbl_employment'
TBL_HISTORY = 'tbl_history'
TBL_LOANS = 'tbl_loans'

In [2]:
# Set Client
client = bigquery.Client(project = PROJECT_ID)
dataset_ref = client.dataset(DATASET_ID , project = PROJECT_ID)
dataset = client.get_dataset(dataset_ref)

In [11]:
# Read Tables
TBL_APPLICATION_REF = dataset_ref.table(TBL_APPLICATION)
table_application = client.get_table(TBL_APPLICATION_REF)
df_application = client.list_rows(table_application).to_dataframe()
print(f'application table:', df_application.shape)

TBL_BORROWS_REF = dataset_ref.table(TBL_BORROWS)
table_borrows = client.get_table(TBL_BORROWS_REF)
df_borrow = client.list_rows(table_borrows).to_dataframe()
print(f'borrow table:', df_borrow.shape)

TBL_EMPLOYMENT_REF = dataset_ref.table(TBL_EMPLOYMENT)
table_employment = client.get_table(TBL_EMPLOYMENT_REF)
df_employment = client.list_rows(table_employment).to_dataframe()
print(f'employment table:', df_employment.shape)

TBL_HISTORY_REF = dataset_ref.table(TBL_HISTORY)
table_history = client.get_table(TBL_HISTORY_REF)
df_history = client.list_rows(table_history).to_dataframe()
print(f'history table:', df_history.shape)

TBL_LOANS_REF = dataset_ref.table(TBL_LOANS)
table_loan = client.get_table(TBL_LOANS_REF)
df_loan = client.list_rows(table_loan).to_dataframe()
print(f'loan table:', df_loan.shape)

application table: (500, 8)
borrow table: (400, 8)
employment table: (400, 7)
history table: (1500, 10)
loan table: (291, 10)


# Feature Engineering

In [14]:
# Aggregate credit history per borrower
credit_agg = df_history.groupby('borrower_id').agg(
    total_accounts       = ('account_type', 'count'),
    avg_account_age_mths = ('account_age_months', 'mean'),
    total_credit_limit   = ('credit_limit', 'sum'),
    total_balance_owed   = ('balance_owed', 'sum'),
    total_late_payments  = ('num_late_payments', 'sum'),
    total_derog_marks    = ('num_derogatory_marks', 'sum'),
    total_inquiries      = ('inquiry_last_6months', 'sum'),
    pct_poor_history     = ('payment_history_score', lambda x: (x == 'Poor').mean())
).reset_index()

credit_agg['utilization_ratio'] = (credit_agg['total_balance_owed'] / credit_agg['total_credit_limit'].replace(0, np.nan)).fillna(0).clip(0, 1)

In [16]:
# Merge all tables
df = (df_loan
      .merge(df_borrow[['borrower_id','age','gender','education_level', 'marital_status','state','credit_score']], on='borrower_id', how='left')
      .merge(df_employment[['borrower_id','employment_type','employment_length_years', 'annual_income','monthly_income']], on='borrower_id', how='left')
      .merge(credit_agg, on='borrower_id', how='left')
      .merge(df_application[['application_id','loan_purpose','interest_rate_offered']], on='application_id', how='left'))

In [20]:
# Engineered features
df['debt_to_income_ratio'] = df['loan_amount'] / (df['annual_income'] / 12).replace(0, np.nan)
df['loan_to_income_ratio'] = df['loan_amount'] / df['annual_income'].replace(0, np.nan)
df['monthly_payment_burden']  = (df['loan_amount'] / df['loan_term_months']) / df['monthly_income'].replace(0, np.nan)
df['credit_score_band'] = pd.cut(df['credit_score'],  bins=[0,580,670,740,800,850],  labels=['Very Poor','Fair','Good','Very Good','Exceptional'])
df['income_per_age'] = df['annual_income'] / df['age'].replace(0, np.nan)
df['derog_per_account'] = df['total_derog_marks'] / df['total_accounts'].replace(0, np.nan)

# Data Preprocessing

In [25]:
TARGET = 'default_flag'

DROP_COLS = ['loan_id','application_id','borrower_id','loan_status',  'origination_date','monthly_payment']

CATEGORICAL_COLS = ['gender','education_level','marital_status','state',  'employment_type','loan_purpose','credit_score_band']

NUMERIC_COLS = ['loan_amount','loan_term_months','interest_rate','age',
                'credit_score','employment_length_years','annual_income',
                'monthly_income','total_accounts','avg_account_age_mths',
                'total_credit_limit','total_balance_owed','total_late_payments',
                'total_derog_marks','total_inquiries','pct_poor_history',
                'utilization_ratio','debt_to_income_ratio','loan_to_income_ratio',
                'monthly_payment_burden','income_per_age','derog_per_account']

X = df.drop(columns=[TARGET] + [c for c in DROP_COLS if c in df.columns])
y = df[TARGET]

In [26]:
# Encode categoricals
for col in CATEGORICAL_COLS:
    if col in X.columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

In [27]:
# Keep only model columns
model_cols = [c for c in NUMERIC_COLS + CATEGORICAL_COLS if c in X.columns]
X = X[model_cols]

In [28]:
# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Train / Validation / Test

In [29]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X_imp, y, test_size=0.15, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

In [30]:
# Class imbalance weight
neg, pos = np.bincount(y_train)
class_weight = {0: 1.0, 1: neg / pos}

N_FEATURES = X_train_sc.shape[1]

# Model Building

In [32]:
def build_model(hp=None):
    if hp:
        units_1  = hp.Int('units_1', 128, 512, step=64)
        units_2  = hp.Int('units_2', 64, 256, step=64)
        units_3  = hp.Int('units_3', 32, 128, step=32)
        dropout  = hp.Float('dropout', 0.2, 0.5, step=0.1)
        lr = hp.Choice('lr', [1e-3, 5e-4, 1e-4])
        l2_reg = hp.Float('l2_reg', 1e-4, 1e-2, sampling='log')
    else:
        units_1, units_2, units_3 = 256, 128, 64
        dropout  = 0.3
        lr = 5e-4
        l2_reg  = 1e-3

    inp = layers.Input(shape=(N_FEATURES,), name='input')

    x = layers.Dense(units_1, kernel_regularizer=regularizers.l2(l2_reg))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Dense(units_2, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Dense(units_3, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Residual-style skip connection
    skip = layers.Dense(units_3)(inp)
    x = layers.Add()([x, skip])
    x = layers.Activation('relu')(x)

    out = layers.Dense(1, activation='sigmoid', name='default_probability')(x)

    model = keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

# Hyperparameter Tunning

In [33]:
try:
    tuner = kt.BayesianOptimization(
        build_model,
        objective=kt.Objective('val_auc', direction='max'),
        max_trials=8,
        directory='/content/drive/MyDrive/project',
        project_name='lendwise_credit_risk',
        overwrite=True
    )
    callbacks_tuner = [
        EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True),
    ]
    tuner.search(X_train_sc, y_train,
                 epochs=30, batch_size=64,
                 validation_data=(X_val_sc, y_val),
                 class_weight=class_weight,
                 callbacks=callbacks_tuner,
                 verbose=0)
    best_hp = tuner.get_best_hyperparameters(1)[0]
    print(f"Best HPs: units_1={best_hp.get('units_1')}, "
          f"units_2={best_hp.get('units_2')}, "
          f"dropout={best_hp.get('dropout'):.2f}, "
          f"lr={best_hp.get('lr')}")
    model = tuner.hypermodel.build(best_hp)
    TUNER_USED = True
except Exception as e:
    print(f"Keras Tuner not available ({e}). Using default architecture.")
    model = build_model()
    TUNER_USED = False

model.summary()

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copied_kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/tuner.py", line 233, in _build_and_fit_model
    results = self.hypermodel.fit(hp, model, *args, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-pa

Keras Tuner not available (Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copied_kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/tuner.py", line 233, in _build_and_fit_model
    results = self.hypermodel.fit(hp, model, *args, **kwargs)
              ^^^^^^

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copied_kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/tuner.py", line 233, in _build_and_fit_model
    results = self.hypermodel.fit(hp, model, *args, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-pa

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 29)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │      7,680 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_4[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 256)       │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ activation_4[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │     32,896 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_5[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 128)       │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ activation_5[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      8,256 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_6[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 64)        │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      1,920 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 64)        │          0 │ activation_6[0][… │
│                     │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 64)        │          0 │ add_1[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ default_probability │ (None, 1)         │         65 │ activation_7[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 52,609 (205.50 KB)

 Trainable params: 51,713 (202.00 KB)

 Non-trainable params: 896 (3.50 KB)

# Training Model

In [35]:
callbacks = [
    EarlyStopping(monitor='val_auc', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
]

history = model.fit(
    X_train_sc, y_train.astype(int),
    epochs=100,
    batch_size=32,
    validation_data=(X_val_sc, y_val.astype(int)),
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/step - accuracy: 0.6029 - auc: 0.6144 - loss: 1.4639 - precision: 0.2529 - recall: 0.5500 - val_accuracy: 0.5263 - val_auc: 0.2949 - val_loss: 1.0211 - val_precision: 0.0769 - val_recall: 0.1429 - learning_rate: 5.0000e-04
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6603 - auc: 0.6732 - loss: 1.4107 - precision: 0.3038 - recall: 0.6000 - val_accuracy: 0.6053 - val_auc: 0.3917 - val_loss: 0.9941 - val_precision: 0.1000 - val_recall: 0.1429 - learning_rate: 5.0000e-04
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6651 - auc: 0.7643 - loss: 1.2434 - precision: 0.3295 - recall: 0.7250 - val_accuracy: 0.6842 - val_auc: 0.4516 - val_loss: 0.9737 - val_precision: 0.2222 - val_recall: 0.2857 - learning_rate: 5.0000e-04
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6986 - auc: 0.8035 - loss: 1.1794 - precision: 0.3614 - recall: 0.7500 - val_accuracy: 0.6842 - val_auc: 0.5415 - val_loss: 0.9493 -

# Evaluation

In [36]:
y_prob_val  = model.predict(X_val_sc, verbose=0).flatten()
y_prob_test = model.predict(X_test_sc, verbose=0).flatten()

# Optimal threshold via Youden's J statistic
fpr, tpr, thresholds = roc_curve(y_val, y_prob_val)
youden_j = tpr - fpr
optimal_idx = np.argmax(youden_j)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal classification threshold: {optimal_threshold:.4f}")

y_pred_test = (y_prob_test >= optimal_threshold).astype(int)

roc_auc = roc_auc_score(y_test, y_prob_test)
acc     = accuracy_score(y_test, y_pred_test)
f1      = f1_score(y_test, y_pred_test)

print(f"\n{'='*50}")
print(f"TEST SET RESULTS")
print(f"{'='*50}")
print(f"ROC-AUC:  {roc_auc:.4f}")
print(f"Accuracy: {acc:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_test, target_names=['Repaid','Defaulted']))

Optimal classification threshold: 0.5724

TEST SET RESULTS
ROC-AUC:  0.9965
Accuracy: 0.9318
F1-Score: 0.8421

Classification Report:
              precision    recall  f1-score   support

      Repaid       1.00      0.92      0.96        36
   Defaulted       0.73      1.00      0.84         8

    accuracy                           0.93        44
   macro avg       0.86      0.96      0.90        44
weighted avg       0.95      0.93      0.94        44



#Apply Prediction on Test and Export to BigQuery

In [37]:
test_loans = df.iloc[y_test.index].copy()
risk_scores_raw = y_prob_test * 100
risk_class = pd.cut(risk_scores_raw,
                     bins=[0, 30, 60, 100],
                     labels=['Low','Medium','High'])

final_predictions = pd.DataFrame({
    'loan_id': test_loans['loan_id'].values,
    'borrower_id':  test_loans['borrower_id'].values,
    'predicted_default_probability': np.round(y_prob_test, 4),
    'risk_score':  np.round(risk_scores_raw, 2),
    'risk_class': risk_class.to_numpy(), # Changed .values to .to_numpy()
    'decision_recommendation': np.where(risk_scores_raw < 30, 'Auto-Approve',
                                    np.where(risk_scores_raw < 60, 'Manual Review', 'Auto-Reject')),
    'actual_default':  y_test.values,
    'model_correct': (y_pred_test == y_test.values).astype(int)
})

print(final_predictions.head(10).to_string(index=False))
print(f"\nRisk Distribution:")
print(final_predictions['risk_class'].value_counts())
print(f"\nModel Accuracy: {final_predictions['model_correct'].mean():.2%}")

loan_id borrower_id  predicted_default_probability  risk_score risk_class decision_recommendation  actual_default  model_correct
LN00209       B0102                         0.1085   10.850000        Low            Auto-Approve               0              1
LN00287       B0292                         0.3362   33.619999     Medium           Manual Review               0              1
LN00099       B0030                         0.9796   97.959999       High             Auto-Reject               1              1
LN00053       B0005                         0.2631   26.309999        Low            Auto-Approve               0              1
LN00281       B0132                         0.8512   85.120003       High             Auto-Reject               1              1
LN00207       B0243                         0.1234   12.340000        Low            Auto-Approve               0              1
LN00168       B0251                         0.2530   25.299999        Low            Auto-Approve

In [39]:
final_predictions.to_gbq ('Bank_Data.credit_predicted' , PROJECT_ID , chunksize=None, if_exists='replace' )

/tmp/ipykernel_21395/1373061905.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_gbq except for the argument 'destination_table' will be keyword-only.
  final_predictions.to_gbq ('Bank_Data.credit_predicted' , PROJECT_ID , chunksize=None, if_exists='replace' )
/tmp/ipykernel_21395/1373061905.py:1: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  final_predictions.to_gbq ('Bank_Data.credit_predicted' , PROJECT_ID , chunksize=None, if_exists='replace' )
100%|██████████| 1/1 [00:00<00:00, 13315.25it/s]
